In [1]:
using Test

In [2]:
function edgeset(edges::Vector)::Set
    "Convert a list of undirected edges to a set of directed edges."
    all_edges = Set()
    for (a,b) in edges
        push!(all_edges, (a, b))
        push!(all_edges, (b, a))
    end
    
    return all_edges
end

edgeset (generic function with 1 method)

In [3]:
function is_matching(
    n::Int,
    edges::Vector,
    matching::Vector
)::Bool
    all_edges = edgeset(edges)
    matched = fill(false, n)
    for (a, b) in matching
        if (a, b) ∉ all_edges || matched[a + 1] || matched[b + 1]
            return false
        end
        matched[a + 1] = true
        matched[b + 1] = true
    end
    
    return true
end

is_matching (generic function with 1 method)

In [4]:
@test is_matching(3, [], []) == true
@test is_matching(3, [(0,1),(1,2)], []) == true
@test is_matching(3, [(0,1),(1,2)], [(0,1)]) == true
@test is_matching(3, [(0,1),(1,2)], [(1,2)]) == true
@test is_matching(3, [(0,1),(1,2)], [(1,0)]) == true
@test is_matching(3, [(0,1),(1,2)], [(2,1)]) == true
@test is_matching(3, [(0,1),(1,2)], [(3,4)]) == false
@test is_matching(3, [(0,1),(1,2)], [(0,0)]) == false
@test is_matching(3, [(0,1),(1,2)], [(2,1),(0,1)]) == false
@test is_matching(3, [(0,1),(1,2)], [(2,1),(2,1)]) == false
@test is_matching(3, [(0,1),(1,2)], [(2,1),(1,2)]) == false
@test is_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(3,2)]) == true
@test is_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(1,2)]) == false
@test is_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(2,0)]) == false

Test Passed

In [5]:
function is_maximal_matching(
    n::Int,
    edges::Vector,
    matching::Vector
)::Bool
    all_edges = edgeset(edges)
    matched = fill(false, n)
    for (a, b) in matching
        if (a, b) ∉ all_edges || matched[a + 1] || matched[b + 1]
            return false
        end
        matched[a + 1] = true
        matched[b + 1] = true
    end
    
    return all(matched[a + 1] || matched[b + 1] for (a, b) in edges)
end

is_maximal_matching (generic function with 1 method)

In [6]:
edges = [(1,2),(1,7),(2,3),(2,4),(3,4),(3,5),(3,7),(4,8),(5,6),(5,0),(6,0),(6,7)]
@test is_maximal_matching(9, edges, [(1,2),(3,4),(5,6)])
@test !is_maximal_matching(9, edges, [(1,2),(3,4)])
@test !is_maximal_matching(9, edges, [(1,2),(3,4),(5,7)])
@test !is_maximal_matching(9, edges, [(1,2),(3,4),(5,0)])

edges8 = [(0,1),(0,2),(0,3),(0,4),(0,5)]
@test [v for v in 1:5 if is_maximal_matching(6, edges8, [(0,v)])] == [1, 2, 3, 4, 5]
@test !is_maximal_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(2,1)])
@test is_maximal_matching(5, [], [])
@test is_maximal_matching(5, [(0,3),(2,4)], [(0,3),(4,2)])
@test is_maximal_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(2,3)])

Test Passed

In [7]:
function is_perfect_matching(
    n::Int,
    edges::Vector,
    matching::Vector
)::Bool
    if n % 2 == 0 && div(n, 2) != length(matching)
        return false
    end
    
    return is_matching(n, edges, matching)
end

is_perfect_matching (generic function with 1 method)

In [8]:
@test !is_perfect_matching(2, [], [])
@test !is_perfect_matching(4, [(0,1),(1,2)], [(3,4)])
@test !is_perfect_matching(2, [(0,1)], [(0,0)])
@test !is_perfect_matching(4, [(0,1),(1,2)], [(2,1),(0,1)])
@test !is_perfect_matching(3, [(0,1),(1,2)], [(2,1),(2,1)])
@test !is_perfect_matching(4, [(0,1),(1,2)], [(2,1),(1,2)])
@test is_perfect_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(0,1),(3,2)])
@test is_perfect_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(2,1),(0,3)])
@test !is_perfect_matching(4, [(0,1),(1,2),(2,3),(3,0)], [(1,2)])

edges8 = [(0,1),(0,2),(0,3),(0,4),(0,5)]
@test [v for v in 0:5 if is_perfect_matching(6, edges8, [(0,v)])] == []

Test Passed

In [9]:
function adjlists(
    n::Int,
    edges::Vector,
    matching_edges::Set
)::Tuple{Vector{Vector}, Vector{Vector}}
    succ_matching = [[] for _ in 1:n]
    succ_nonmatching = [[] for _ in 1:n]
    for (a, b) in edges
        s = (a, b) ∈ matching_edges ? succ_matching : succ_nonmatching
        push!(s[a + 1], b)
        push!(s[b + 1], a)
    end
    
    return succ_matching, succ_nonmatching
end

adjlists (generic function with 1 method)

In [10]:
function find_augmenting_path(
    n::Int,
    edges::Vector,
    matching::Vector
)::Vector
    free_vertices = fill(true, n)
    for (a, b) in matching
        free_vertices[a + 1] = free_vertices[b + 1] = false
    end
    succ_matching, succ_nonmatching = adjlists(n, edges, edgeset(matching))
    seen = fill(false, n)

    # DFS-based exploration    
    function rec(start::Int)
        seen[start + 1] = true
        for y in succ_nonmatching[start + 1]
            if seen[y + 1]
                continue
            elseif free_vertices[y + 1]
                return [start, y]
            end

            seen[y + 1] = true
            for z in succ_matching[y + 1]
                if seen[z + 1]
                    continue
                end
                
                res = rec(z)
                if !isnothing(res)
                    return [[start, y]; res]
                end
            # Erase "seen" on backtrack
            # This unfortunately causes the algorithm to become exponential 
            # in the worst case since it explores all possibles alternating paths.
            # For a polynomial implementation lookup Edmonds' Blossom algorithm.
            seen[y + 1] = false
            end
        end
        
        seen[start + 1] = false
        return nothing
    end
        
    for (start, is_free) in enumerate(free_vertices)
        if is_free
            res = rec(start - 1)
            if !isnothing(res)
                return res
            end
        end
    end

    return []
end

find_augmenting_path (generic function with 1 method)

In [16]:
matching1 = [(1,2),(3,4),(5,7)]
edges_2_3 = [(1,2),(1,6),(2,3),(2,4),(3,4),(3,5),(3,6),(4,0),(5,7),(5,0),(6,7),(7,0)]
@test find_augmenting_path(8, edges_2_3, matching1) == [0, 4, 3, 2, 1, 6]

matching2 = [(1,6),(2,4),(7,0),(5,3)]
@test find_augmenting_path(8, edges_2_3, matching2) == []

matching3 = [(2,3),(0,1)]
edges3 = [(0,1),(1,2),(0,2),(3,2),(3,4),(4,5),(5,3)]
@test find_augmenting_path(6, edges3, matching3) == [4, 5]	

matching4 = [(0,2),(3,5)]
@test find_augmenting_path(6, edges3, matching4) == [1, 0, 2, 3, 5, 4]

matching5 = []
@test find_augmenting_path(6, edges3, matching5) == [0, 1]

edges6 = [(15,16),(16,1),(14,4),(0,1),(1,2),(2,3),(3,4),
        (4,5),(5,6),(6,7),(7,8),(8,9),(9,10),(10,11),
        (10,12),(12,13),(13,8),(7,14),(3,15)]
matching6 = [(0,1),(16,15),(3,4),(5,6),(7,8),(9,10),(13,12)]
@test find_augmenting_path(17, edges6, matching6) == [2, 3, 4, 14]

edges7 = [(11, 12), (14, 3), (16, 12), (10, 5), (0, 8), (1, 4), (2, 15), (4, 14), (8, 13),
        (5, 9), (7, 1), (14, 6), (13, 2), (15, 0), (5, 13), (12, 10), (6, 7), (9, 16), (0, 1)]
matching7 = [(3, 14), (2, 15), (1, 4), (6, 7), (5, 9), (0, 8), (10, 12)]
@test find_augmenting_path(17, edges7, matching7) == [11, 12, 10, 5, 9, 16]

edges8 = [(0, 1), (1, 2), (2, 3), (2, 4), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (13, 15), (14, 15), (15, 16), (16, 17)]
matching8 = [(1, 2), (3, 6), (4, 5), (7, 8), (9, 10), (11, 13), (12, 14), (15, 16)]
@test find_augmenting_path(18, edges8, matching8) == [0, 1, 2, 3, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17]

edges9 = [(0, 1), (1, 2), (2, 4), (2, 3), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (13, 15), (14, 15), (15, 16), (16, 17)]
matching9 = [(1, 2), (3, 6), (4, 5), (7, 8), (9, 10), (11, 13), (12, 14), (15, 16)]
@test find_augmenting_path(18, edges9, matching9) == [0, 1, 2, 3, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17]

edges10 = [(0, 1), (1, 2), (2, 3), (2, 4), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (14, 15), (13, 15), (15, 16), (16, 17)]
matching10 = [(1, 2), (3, 6), (4, 5), (7, 8), (9, 10), (11, 13), (12, 14), (15, 16)]
@test find_augmenting_path(18, edges10, matching10) == [0, 1, 2, 3, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17]

edges11 = [(0, 1), (1, 2), (2, 4), (2, 3), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (14, 15), (13, 15), (15, 16), (16, 17)]
matching11 = [(1, 2), (3, 6), (4, 5), (7, 8), (9, 10), (11, 13), (12, 14), (15, 16)]
@test find_augmenting_path(18, edges11, matching11) == [0, 1, 2, 3, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17]

path = find_augmenting_path(4, [(0,1),(1,2),(3,2)], [(2,1)])
@test path == [0,1,2,3] || path == [3,2,1,0]

Test Passed

In [12]:
a = Set(1:10)
b = Set(1:2:10)
println(a, b)
collect(symdiff(a, b))

Set([5, 4, 6, 7, 2, 10, 9, 8, 3, 1])Set([5, 7, 9, 3, 1])


5-element Vector{Int64}:
  4
  6
  2
 10
  8

In [13]:
function normedge(e::Tuple{Int, Int})::Tuple{Int, Int}
    """Normalize edges so that vertices are increasing"""
    return e[1] < e[2] ? e : reverse(e)
end

function update_matching(matching::Vector, augpath::Vector)::Vector
    m = Set(normedge(e) for e in matching)
    p = Set(normedge(e) for e in zip(augpath, augpath[2:length(augpath)]))
    return collect(symdiff(m, p))
end
    
function find_maximum_matching(n::Int, edges::Vector)::Vector
    matching = []
    p = find_augmenting_path(n, edges, matching)
    while p != []
        matching = update_matching(matching, p)
        p = find_augmenting_path(n, edges, matching)
    end
    return matching
end

find_maximum_matching (generic function with 1 method)

In [14]:
function valid_path(
    n::Int,
    edges::Vector,
    matching::Vector
)::Bool
    e = Set(edges)
    seen = fill(false, n)
    for (x, y) in matching[2:length(matching)]
        if seen[x + 1] || seen[y + 1] || ((x, y) ∉ e && (y, x) ∉ e)
            return false
        end

        seen[x + 1] = seen[y + 1] = true
    end
    
    return true
end

valid_path (generic function with 1 method)

In [15]:
edges_2_3 = [(1,2),(1,6),(2,3),(2,4),(3,4),(3,5),(3,6),(4,0),(5,7),(5,0),(6,7),(7,0)]
m0 = find_maximum_matching(8, edges_2_3)
@test valid_path(8, edges_2_3, m0) && length(m0) == 4
@test find_maximum_matching(1,[]) == []
@test find_maximum_matching(2,[]) == []

n1 = 17
edges1= [(11, 12), (14, 3), (16, 12), (10, 5), (0, 8), (1, 4), (2, 15), (4, 14), (8, 13), (5, 9),
         (7, 1), (14, 6), (13, 2), (15, 0), (5, 13), (12, 10), (6, 7), (9, 16), (0, 1)]
m1 = find_maximum_matching(n1, edges1)
@test valid_path(n1, edges1, m1) && length(m1) == 8

n2 = 18
edges2 = [(0, 1), (1, 2), (2, 3), (2, 4), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (13, 15), (14, 15), (15, 16), (16, 17)]
m2 = find_maximum_matching(n2, edges2)
@test valid_path(n2, edges2, m2) && length(m2) == 9

n3 = 18
edges3 = [(0, 1), (1, 2), (2, 4), (2, 3), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (13, 15), (14, 15), (15, 16), (16, 17)]
m3 = find_maximum_matching(n3, edges3)
@test valid_path(n3, edges3, m3) && length(m3) == 9

n4 = 18
edges4 = [(0, 1), (1, 2), (2, 3), (2, 4), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (14, 15), (13, 15), (15, 16), (16, 17)]
m4 = find_maximum_matching(n4, edges4)
@test valid_path(n4, edges4, m4) && length(m4) == 9

n5 = 18
edges5 = [(0, 1), (1, 2), (2, 4), (2, 3), (3, 6), (4, 5), (5, 6), (6, 7),
              (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (11, 13), (12, 14),
              (14, 15), (13, 15), (15, 16), (16, 17)]
m5 = find_maximum_matching(n5, edges5)
@test valid_path(n5, edges5, m5) && length(m5) == 9

Test Passed